# Week 8 Exercise — The Price is Right (Deal Hunting UI)

**Week 8 finale (simplified):** A **Gradio UI** for "The Price is Right" deal-hunting: table of deals with Description, Price, Estimate, Discount, URL. Uses **OpenAI only** (no Modal or course agents).

- **Surface deals:** Pre-loaded example products; GPT estimates fair price; discount = list price − estimate.
- **Add deal:** Enter a product description, list price, and URL to add a row.
- Run all cells, then launch the UI.

In [ ]:
import os
import re
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"OpenAI API key loaded (starts with {api_key[:8]}...)")
else:
    print("OPENAI_API_KEY not set")

MODEL = "gpt-4o-mini"
client = OpenAI() if api_key else None

In [ ]:
def _messages_for(description: str):
    return [{"role": "user", "content": f"Estimate the price of this product in USD. Respond with only the number, no explanation.\n\n{description}"}]

def _parse_price(value):
    if isinstance(value, (int, float)):
        return float(value)
    s = str(value).replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else 0.0

def estimate_price(description: str) -> float:
    if not client or not (description or "").strip():
        return 0.0
    try:
        r = client.chat.completions.create(model=MODEL, messages=_messages_for(description.strip()), temperature=0)
        return _parse_price(r.choices[0].message.content or "0")
    except Exception:
        return 0.0

In [ ]:
EXAMPLE_DEALS = [
    ("Wireless Bluetooth Earbuds, 24hr battery, noise cancelling", 79.99, "https://example.com/earbuds"),
    ("Stainless steel water bottle 32oz insulated", 34.99, "https://example.com/bottle"),
    ("Running shoes men's size 10 breathable mesh", 129.00, "https://example.com/shoes"),
]

def surface_deals() -> list:
    rows = []
    for desc, list_price, url in EXAMPLE_DEALS:
        est = estimate_price(desc)
        discount = max(0.0, round(list_price - est, 2))
        rows.append([desc, list_price, est, discount, url])
    return rows

def add_deal(desc: str, list_price: float, url: str, existing_rows: list) -> tuple:
    if not (desc and str(desc).strip()):
        return existing_rows if existing_rows else [], ""
    try:
        price_val = float(list_price) if list_price is not None else 0.0
    except (TypeError, ValueError):
        price_val = 0.0
    url_str = (url or "").strip() or "—"
    est = estimate_price(desc.strip())
    discount = max(0.0, round(price_val - est, 2))
    new_row = [desc.strip(), price_val, est, discount, url_str]
    updated = list(existing_rows) + [new_row]
    return updated, "Added."

In [ ]:
with gr.Blocks(title="The Price is Right", fill_width=True, css="footer{display:none !important}") as ui:
    gr.Markdown("<div style='text-align: center; font-size: 24px;'>The Price is Right — Deal Hunting Agentic AI</div>")
    gr.Markdown("<div style='text-align: center; font-size: 14px;'>Estimate fair price with OpenAI; discount = list price − estimate. (OpenAI-only simplified version.)</div>")

    opportunities = gr.State(value=[])
    tbl = gr.Dataframe(
        headers=["Description", "Price", "Estimate", "Discount", "URL"],
        label="Deals",
        interactive=False,
        wrap=True,
    )
    msg = gr.Textbox(label="Status", interactive=False)

    def on_surface():
        rows = surface_deals()
        return rows, rows, "Deals surfaced."

    def on_add(desc, price, url, rows):
        updated, status = add_deal(desc, price, url, rows)
        return updated, updated, status

    gr.Button("Surface deals").click(
        fn=on_surface,
        inputs=[],
        outputs=[opportunities, tbl, msg],
    )
    with gr.Row():
        desc_in = gr.Textbox(placeholder="Product description", label="Description", lines=2)
        price_in = gr.Number(label="List price", value=0)
        url_in = gr.Textbox(placeholder="https://...", label="URL")
    gr.Button("Add deal").click(
        fn=on_add,
        inputs=[desc_in, price_in, url_in, opportunities],
        outputs=[opportunities, tbl, msg],
    )

ui.launch(show_api=False)